In [1]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from imblearn.under_sampling import RandomUnderSampler
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sklearn.metrics import classification_report
from torch.autograd import detect_anomaly
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import google.generativeai as genai
from google.generativeai import types
import os
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import google.api_core.exceptions
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

/home/cs/grad/islams32/dev/project/academic/technical-debt/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [3]:
from sklearn.linear_model import LogisticRegression
class EmbeddedLogisticRegression:
    def __init__(self, model_name: str, uri:str, sentence_transformer):
        self.model_name = model_name
        self.model = LogisticRegression()
        self.uri = uri
        self.sentence_transformer = sentence_transformer

    def fit(self, x_train, y_train):
        x_train_encoded = self.sentence_transformer.encode(x_train)
        self.model.fit(x_train_encoded, y_train)

    def predict(self, x_test):
        x_test_encoded = self.sentence_transformer.encode(x_test)
        return self.model.predict(x_test_encoded)

In [4]:
# df = pd.read_csv('../data/maldonado_corrected.csv')
# df['label'] = df['satd_orig'].apply(lambda x: 'yes' if x == 1 else 'no')
# df['text'] = df['comment_text']
# df = df[["text", "label"]]
#
# under_sampler = RandomUnderSampler(sampling_strategy=1, random_state=42)
# X_resampled, y_resampled = under_sampler.fit_resample(df[['text']], df['label'])
#
# # Create balanced DataFrame
# df = pd.DataFrame({'text': X_resampled['text'], 'label': y_resampled})
# dataset = Dataset.from_pandas(df).train_test_split(test_size=0.03, seed=42)
# dataset = dataset.remove_columns(['__index_level_0__'])
# dataset

In [5]:
# df = pd.read_csv('../data/td_comment.csv')
# df['label'] = df['is_td'].apply(lambda x: 'yes' if x == 1 else 'no')
# df = df[["text", "label"]]
#
# under_sampler = RandomUnderSampler(sampling_strategy=1, random_state=42)
# X_resampled, y_resampled = under_sampler.fit_resample(df[['text']], df['label'])
#
# # Create balanced DataFrame
# df = pd.DataFrame({'text': X_resampled['text'], 'label': y_resampled})
# dataset = Dataset.from_pandas(df).train_test_split(test_size=0.2, seed=42)
# dataset = dataset.remove_columns(['__index_level_0__'])
# dataset

In [6]:
# df = pd.read_csv('../data/td_comment.csv')
# df['label'] = df['is_td'].apply(lambda x: 'yes' if x == 1 else 'no')
# df = df[["text", "label"]]
#
# under_sampler = RandomUnderSampler(sampling_strategy=1, random_state=42)
# X_resampled, y_resampled = under_sampler.fit_resample(df[['text']], df['label'])
#
# # Create balanced DataFrame
# df = pd.DataFrame({'text': X_resampled['text'], 'label': y_resampled})
# dataset = Dataset.from_pandas(df).train_test_split(test_size=0.2, seed=42)
# dataset = dataset.remove_columns(['__index_level_0__'])
# dataset

# SATD Detection Dataset

In [7]:
detect_train_df = pd.read_csv('../data/detect_train.csv')
detect_train_dataset = Dataset.from_pandas(detect_train_df)
detect_test_df = pd.read_csv('../data/detect_test.csv')
detect_test_dataset = Dataset.from_pandas(detect_test_df)
detect_train_balanced_df = pd.read_csv('../data/detect_train_balanced.csv')
detect_train_balanced_dataset = Dataset.from_pandas(detect_train_balanced_df)

# SATD Classification Dataset

In [8]:
classify_train_df = pd.read_csv('../data/classify_train.csv')
classify_train_dataset = Dataset.from_pandas(classify_train_df)
classify_test_df = pd.read_csv('../data/classify_test.csv')
classify_test_dataset = Dataset.from_pandas(classify_test_df)

# N-Shots SATD Detection Dataset

In [9]:
n_shot_detect_df = pd.read_csv('../data/n_shot_detect.csv')
n_shot_detect_dataset = Dataset.from_pandas(n_shot_detect_df)
n_shot_detect_dataset

Dataset({
    features: ['id', 'repository', 'text', 'label', 'code_before', 'code_after', 'cot'],
    num_rows: 6
})

In [10]:
class PromptTemplate:
    def __init__(self, name, definition, instruction, n_shot_template, line_m_before, line_n_after):
        self._name = name
        self._definition = definition
        self._instruction = instruction
        self._n_shot_template = n_shot_template
        self._line_m_before = line_m_before
        self._line_n_after = line_n_after

    @property
    def name(self):
        return self._name

    @property
    def definition(self):
        return self._definition

    @property
    def instruction(self):
        return self._instruction

    @property
    def line_m_before(self):
        return self._line_m_before

    @property
    def line_n_after(self):
        return self._line_n_after

    @property
    def shot_template(self):
        return self._n_shot_template

    def __repr__(self):
        return f"PromptTemplate(name={self.name}, description='{self.definition}', example='{self.shot_template}')"

In [11]:
class ModelConfig:
    def __init__(self, name: str, architecture: str, uri: str):
        self.name = name
        self.architecture = architecture
        self.uri = uri

    def __repr__(self):
        return f"ModelConfig(name='{self.name}', uri='{self.uri}')"

In [12]:
class ChatGpt4Model:
    def __init__(self, model_name: str, client: OpenAI):
        self.model_name = model_name
        self.client = client

    def generate(self, prompt):
        completion = self.client.chat.completions.create(
            model=self.model_name,
            store=True,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return completion.choices[0].message.content.strip().split()[-1].lower()


In [13]:
import jpype
import jpype.imports
from jpype.types import *
from dotenv import load_dotenv
import os
load_dotenv()


class TextMiningBasedSatdDetector:
    def __init__(self, model_name: str):
        self.model_name = model_name
    def fit(self, x_train, y_train):
        pass
    def predict(self, x_test):
        if not jpype.isJVMStarted():
            jar_path = os.getenv('SATD_DETECTOR_JAR')
            dependency_path= os.getenv('SATD_DETECTOR_DEPENDENCY')
            jvm_args = ["-Xss512m"]
            jpype.startJVM(jpype.getDefaultJVMPath(),  classpath=[jar_path, dependency_path], )
        from satd_detector.core.utils import SATDDetector
        detector1 = SATDDetector()
        y_pred = []
        for comment in x_test:
            if detector1.isSATD(comment):
                y_pred.append('yes')
            else:
                y_pred.append('no')
        return y_pred



In [14]:
import ast
from util import sha1
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

@retry(
    stop=stop_after_attempt(10),  # Stop after 5 retries
    wait=wait_exponential(multiplier=2, min=60, max=2 * 60),
    retry=retry_if_exception_type(google.api_core.exceptions.ResourceExhausted))
def create_embedding(uri, text):
    return genai.embed_content(
                        model=uri,
                        content=text,
                        task_type="classification")
class GeminiSentenceTransformer:
    def __init__(self, uri: str, use_cache=False):
        self.uri = uri
        self.use_cache = use_cache
        self.file = f'./cache/{uri.split("/")[-1]}.csv'
        if not os.path.exists(self.file):
              with open(self.file, "w") as file:
                file.write("text,hash,embedding")
                file.flush()
        self.cache_df = pd.read_csv(self.file)
        self.encoding_map = {r['hash'] : np.array(ast.literal_eval(r['embedding']), dtype=np.float32) for row_index, r in self.cache_df.iterrows()}
    def encode(self, features):
        encoded_features = []
        rows = []
        try:
            for text in features:
                if self.use_cache:
                    hash = sha1(text)
                    # embedding_df = self.cache_df[self.cache_df['hash'] == hash]['embedding']
                    if hash not in self.encoding_map:
                    # if embedding_df.empty:
                        response = create_embedding(self.uri, text)
                        embedding_value = response["embedding"]
                        rows.append([text, hash, embedding_value])
                        # self.cache_df.loc[len(self.cache_df)] = [text, hash, embedding_value]
                        # self.cache_df.to_csv(self.file, index=False)
                        # self.cache_df = pd.read_csv(self.file)
                        self.encoding_map[hash] = np.array(embedding_value)
                    encoded_features.append(self.encoding_map[hash])

                    # else:
                        # encoded_features.append(np.array(ast.literal_eval(embedding_df.iloc[0]), dtype=np.float32))
                else:
                    raise Exception('Not Implemented Yet')
        except Exception as e:
            raise e
        finally:
            self.checkpoint(rows)
        return encoded_features
    def checkpoint(self, rows):
        if len(rows) > 0:
            new_df = pd.DataFrame(rows, columns=self.cache_df.columns)
            pd.concat([self.cache_df, new_df]).to_csv(self.file, index=False)
            self.cache_df = pd.read_csv(self.file)


In [15]:
from util import get_first_n_line, get_last_n_line


def create_prompt(prompt_template, n_shot_x, n_shot_y, n_shot_code_before, n_shot_code_after, n_shot_cot, question,
                  code_before, code_after, tokenizer):
    instances = []
    for index, [x, y, cb, ca, cot] in enumerate(zip(n_shot_x, n_shot_y, n_shot_code_before, n_shot_code_after, n_shot_cot)):
        example_text = prompt_template.shot_template.format(**{'comment': x,
                                                             'label': y,
                                                             'code_before': get_last_n_line(cb, prompt_template.line_m_before), 'code_after': get_first_n_line(ca, prompt_template.line_n_after),
                                                               'cot': cot})
        instances.append(example_text)

    instances.append(prompt_template.shot_template.format(**{'comment': question,
                                                           'label': '',
                                                           'code_before': get_last_n_line(code_before, prompt_template.line_m_before), 'code_after': get_first_n_line(code_after, prompt_template.line_n_after), 'cot': ''}))
    prompt_text = prompt_template.definition + "\n" + prompt_template.instruction + "\n" + "\n" + "\n\n".join(instances)
    return prompt_text


In [16]:
# from util import get_first_n_line, get_last_n_line
# def get_token_length(tokenizer, text):
#     if tokenizer:
#         tokens = tokenizer(text, return_tensors='pt')
#         return tokens['input_ids'].shape[1]
#     else:
#         return len(text)
#
#
# def create_prompt(prompt_template, x, y, n_shot_code_before, n_shot_code_after, question, code_before, code_after,
#                   tokenizer):
#     model_max_length = tokenizer.model_max_length if tokenizer else float('inf')
#     initial_text = prompt_template.definition + "\n" + prompt_template.instruction
#     allocated_tokens = len(x) + 5  # Formatting
#     allocated_tokens += get_token_length(tokenizer, initial_text)
#     question_prefix = question
#     while len(question_prefix) > 0:
#         target_sample_token_length = get_token_length(tokenizer,
#                                                       prompt_template.shot_template.format(question_prefix, '', get_last_n_line(code_before, prompt_template.line_m_before), get_first_n_line(code_after, prompt_template.line_n_after)))
#         if allocated_tokens + target_sample_token_length <= model_max_length:
#             allocated_tokens += target_sample_token_length
#             break
#         else:
#             question_prefix = question_prefix[: len(question_prefix) // 2]
#
#     instances = []
#     skipped = 0
#     for index, [x, y, cb, ca] in enumerate(zip(x, y, n_shot_code_before, n_shot_code_after)):
#         example_text = prompt_template.shot_template.format(x, y, get_last_n_line(cb, prompt_template.line_m_before), get_first_n_line(ca, prompt_template.line_n_after))
#         example_token_length = get_token_length(tokenizer, example_text)
#         if allocated_tokens + example_token_length < model_max_length:
#             instances.append(prompt_template.shot_template.format(x, y, get_last_n_line(cb, prompt_template.line_m_before), get_first_n_line(ca, prompt_template.line_n_after)))
#             allocated_tokens += example_token_length
#         else:
#             skipped += 1
#     if skipped > 0:
#         print(f'Skipping {skipped} shots due to token limits')
#
#     instances.append(prompt_template.shot_template.format(question_prefix, '', get_last_n_line(code_before, prompt_template.line_m_before), get_first_n_line(code_after, prompt_template.line_n_after)))
#     prompt_text = initial_text + "\n" + "\n" + "\n\n".join(instances)
#     prompt_token_length = get_token_length(tokenizer, prompt_text)
#     if prompt_token_length > model_max_length:
#         print(f'prompt length {prompt_token_length}')
#     return prompt_text

In [17]:
def create_model_and_tokenizer(model_config: ModelConfig):
    is_hf_model = True
    uri = model_config.uri
    if 'sentence-embedded-regression' in model_config.architecture.lower():
        return EmbeddedLogisticRegression(model_config.name, model_config.uri, SentenceTransformer(uri)), None
    elif 'gemini-embedded-regression' in model_config.architecture.lower():
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
        return EmbeddedLogisticRegression(model_config.name, model_config.uri, GeminiSentenceTransformer(uri, True)), None
    elif 'text-mining' in model_config.architecture.lower():
        return TextMiningBasedSatdDetector(model_config.uri), None
    elif 'gpt-4' in model_config.architecture:
        return ChatGpt4Model(model_config.uri, OpenAI(api_key=os.getenv("OPEN_AI_API_KEY"))), None
    if 'gpt' in model_config.architecture.lower():
        model = AutoModelForCausalLM.from_pretrained(uri)
    elif "/bert" in model_config.architecture.lower() or "/codebert" in model_config.architecture.lower():
        model = AutoModelForSequenceClassification.from_pretrained(uri, num_labels=2)
    elif 'gemini' in model_config.architecture.lower():
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
        model = genai.GenerativeModel(uri)
        is_hf_model = False
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(uri)
    if is_hf_model:
        model.to(device)
        tokenizer = AutoTokenizer.from_pretrained(model_config.uri, use_fast=True)
    else:
        tokenizer = None
    return model, tokenizer


In [18]:
from enum import Enum


class FewShotSelectionStrategy(Enum):
    RANDOM = 'random'
    SIMILAR = 'similar'
    MANUAL_CRAFTED = 'manual_crafted'


In [19]:
import random


def pick_n_shot(train_dataset, x, y, index, st_similarity, n=0, strategy=None):
    if len(x) < n:
        raise Exception(f'only {len(x)} examples available for {n} shots')
    indexes = []
    if strategy == FewShotSelectionStrategy.RANDOM:
        indexes = random.sample(range(len(x)), n)
    elif strategy == FewShotSelectionStrategy.SIMILAR:
        _, top_n_indices = st_similarity[index].topk(n)
        indexes.extend(top_n_indices.tolist())
    elif strategy == FewShotSelectionStrategy.MANUAL_CRAFTED:
        indexes = [i for i in range(n)]
    shot_x = []
    shot_y = []
    code_before = []
    code_after = []
    cot = []
    for index in indexes:
        shot_x.append(x[index])
        shot_y.append(y[index])
        code_before.append(train_dataset['code_before'][index])
        code_after.append(train_dataset['code_after'][index])
        cot.append(train_dataset['cot'][index])
    return shot_x, shot_y, code_before, code_after, cot


In [20]:
sentence_transformer = SentenceTransformer('all-MiniLM-L6-v2')


In [21]:
def print_classification(test_x, test_y, y_pred):
    print("Incorrect Predictions:")
    for text, true, pred in zip(test_x, test_y, y_pred):
        if true != pred:
            print(f"✖ {text} (Label: {true}, Predicted: {pred})")


In [22]:
# PROMPT_TEMPLATES = [
#     PromptTemplate(
#         name="With Indicators",
#         definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that solely describe expected behavior, general actions, testing actions or issue references unless they explicitly acknowledge it requires future work.",
#         instruction="Classify whether the comment contains SATD (yes/no)",
#         n_shot_template="<EXAMPLE>\nComment: {}\nLabel: {}\n</EXAMPLE>"
#     ),
#     PromptTemplate(
#         name="No Keywords",
#         definition="Self-admitted technical debt (SATD) is technical debt admitted by the developer through source code comments.",
#         instruction="Assign the label of yes or no for each given source code comment",
#         n_shot_template="Comment: {}\nLabel: {}"
#     ),
#     PromptTemplate(
#         name="MAT Keywords",
#         definition="Self-admitted technical debt (SATD) are technical debt admitted by the developer through source code comments. SATD comments usually  contain specific keywords: TODO, FIXME, HACK, and XXX.",
#         instruction="Assign the label of yes or no for each given source code comment.",
#         n_shot_template="Comment: {}\nLabel: {}"
#     ), PromptTemplate(
#         name="Jitterbug Keywords",
#         definition="Self-admitted technical debt (SATD) is technical debt admitted by the developer through source code comments. SATD comments usually contain specific keywords: TODO, FIXME, HACK, and Workaround.",
#         instruction="Assign the label of yes or no for each given source code comment.",
#         n_shot_template="Comment: {}\nLabel: {}"
#     ), PromptTemplate(
#         name="GPT4 Keywords",
#         definition="Self-admitted technical debt (SATD) is technical debt admitted by the developer through source code comments. SATD comments usually contain specific keywords: TODO, FIXME, HACK, XXX, NOTE, DEBT, REFACTOR, OPTIMIZE, TEMP, WORKAROUND, KLUDGE, REVIEW, NOFIX, PENDING, and BUG.",
#         instruction="Assign the label of yes or no for each given source code comment.",
#         n_shot_template="Comment: {}\nLabel: {}"
#     ), PromptTemplate(
#         name="TD",
#         definition="Technical debt (TD) in code comment are comments that that indicates weak code or something need to be done. TD comments usually  contain specific keywords: TODO, FIXME, HACK, and XXX.",
#         instruction="Assign the label of yes or no for each given source code comment.",
#         n_shot_template="Comment: {}\nLabel: {}"
#     )]

In [23]:
FEW_SHOT_SIZES = [0, 1, 2, 3, 5, 10, 15, 20]
MODEL_CONFIG_GEMINI_2_FLASH = ModelConfig(name="Gemini 2 Flash", architecture="gemini", uri="models/gemini-2.0-flash")
MODEL_CONFIG_SENTENCE_EMBEDDED_LR = ModelConfig(name="Sentence Embedded Logistic Regression", architecture="sentence-embedded-regression", uri="all-MiniLM-L6-v2")
MODEL_CONFIG_GEMINI_EMBEDDED_LR = ModelConfig(name="Gemini Embedded Logistic Regression", architecture="gemini-embedded-regression", uri='models/gemini-embedding-exp-03-07')
MODEL_CONFIG_TEXT_MINING_SATD_DETECTOR = ModelConfig(name="Text Mining SATD Detector", architecture="text-mining", uri="text-mining/satd-detector")
MODEL_CONFIGS = [
    MODEL_CONFIG_GEMINI_2_FLASH,
    ModelConfig(name="Chat GPT 4o mini", architecture="gpt-4", uri="gpt-4o-mini"),
    ModelConfig(name="Chat GPT 4o", architecture="gpt-4", uri="gpt-4o"),
    ModelConfig(name="Flan T5 Small", architecture='flan-t5', uri="google/flan-t5-small"),
    ModelConfig(name="Flan T5 Base", architecture='flan-t5', uri="google/flan-t5-base"),
    ModelConfig(name="Flan T5 Large", architecture='flan-t5', uri="google/flan-t5-large"),
    ModelConfig(name="Flan T5 XL", architecture='flan-t5', uri="google/flan-t5-xl"),
    ModelConfig(name="BERT Base", architecture="bert", uri="google-bert/bert-base-uncased"),
    ModelConfig(name="CodeBERT Base", architecture="codebert", uri="microsoft/codebert-base"),
    ModelConfig(name="Facebook BART Base", architecture="bart", uri="facebook/bart-base")

]
FEW_SHOT_STRATEGIES = [FewShotSelectionStrategy.MANUAL_CRAFTED, FewShotSelectionStrategy.RANDOM,
                       FewShotSelectionStrategy.SIMILAR]

In [24]:
def predict_with_prompt(model, tokenizer, prompt):
    if isinstance(model, ChatGpt4Model):
        return model.generate(prompt)
    elif tokenizer:
        inputs = tokenizer(prompt, return_tensors='pt')
        inputs = {key: value.to(device) for key, value in inputs.items()}
        output = tokenizer.decode(
            model.generate(
                inputs["input_ids"],
                max_new_tokens=50
                # generation_config=GenerationConfig(max_new_tokens=5, do_sample=True, temperature=0.01)
            )[0],
            skip_special_tokens=True
        )
        return output.strip()
    else:
        return predict_with_gemini(model, prompt)


@retry(
    stop=stop_after_attempt(10),  # Stop after 5 retries
    wait=wait_exponential(multiplier=2, min=60, max=2 * 60),
    retry=retry_if_exception_type(google.api_core.exceptions.ResourceExhausted),  # Retry on rate limit errors
)
def predict_with_gemini(model, prompt):
    generation_config = types.GenerationConfig(
        temperature=0.0

    )
    return model.generate_content(contents=prompt, generation_config=generation_config).text.split()[-1].lower()


In [89]:
from datetime import datetime
LAST_RUNNING_FILE = None


def detect_satd(task_type, model_config, few_shot_size, prompt_template, few_shot_strategy, dataset, text_column='text',
                label_column='label', verbose=False):
    train_dataset = dataset["train"]
    train_x = train_dataset[text_column]
    train_y = train_dataset[label_column]
    test_dataset = dataset["test"]
    test_x = test_dataset[text_column]
    test_y = test_dataset[label_column]
    test_x_code_before = test_dataset['code_before']
    test_x_code_after = test_dataset['code_after']
    st_similarities = cos_sim(sentence_transformer.encode(test_x),
                                                  sentence_transformer.encode(train_x)) if FewShotSelectionStrategy.SIMILAR == few_shot_strategy else None
    model, tokenizer = create_model_and_tokenizer(model_config)
    y_pred = []
    unseen_labels = []
    running_config = f'{model_config.uri.split("/")[-1]}'
    print(f'Running {running_config}')
    if isinstance(model, EmbeddedLogisticRegression) or isinstance(model, TextMiningBasedSatdDetector):
        model.fit(train_x, train_y)
        y_pred = model.predict( test_x)
    else:
        for i, [text, label, code_before, code_after] in enumerate(zip(test_x, test_y, test_x_code_before, test_x_code_after)):
            shot_x, shot_y, n_shot_code_before, n_shot_code_after, n_shot_cot = pick_n_shot(train_dataset, train_x, train_y, i, st_similarities, few_shot_size,
                                         few_shot_strategy)
            prompt = create_prompt(prompt_template, shot_x, shot_y, n_shot_code_before, n_shot_code_after, n_shot_cot,
                                   text, code_before, code_after, tokenizer)
            pred = predict_with_prompt(model, tokenizer, prompt).lower()
            if verbose:
                print(f'Prompt\n {prompt}')
                if pred != label:
                    print(f'{pred} {label}: {text}')
            if pred not in ['yes', 'no']:
                unseen_labels.append(pred)
                pred = 'no'
            y_pred.append(pred)
    test_output = dataset['test'].to_dict()
    test_output[label_column + '_pred'] = y_pred
    print(classification_report(test_y, y_pred, zero_division=0, digits=3))
    if verbose:
        print(f'Unknown classification count {len(unseen_labels)}')
        print(f'Unknown classification  {unseen_labels}')
    timestamp = datetime.now().strftime("%B %d, %Y, %H:%M:%S")
    global LAST_RUNNING_FILE
    LAST_RUNNING_FILE = f'./cache/{timestamp}#{task_type}_{running_config}.csv'

    Dataset.from_dict(test_output).to_pandas().to_csv(LAST_RUNNING_FILE, index = False)
    # return test_y, y_pred

In [ ]:
# template = PromptTemplate(
#         name="Manually Crafted",
#         definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
#         instruction="Classify whether the comment contains SATD if the confidence level is high(yes/no)",
#         n_shot_template="<EXAMPLE>\nComment: {comment}\nLabel: {label}\nCode Before Comment: {code_before}\nCode After Comment: {code_after}\n</EXAMPLE>",
#         line_m_before=3,
#         line_n_after=3
#     )


In [27]:
def input_subset_df(data_frame):
    from_row_id, to_row_id = [int(index) if index else None for index in input('Row ID').strip().split(':')]
    return data_frame[from_row_id:to_row_id]

# Detect with Gemini 2.0 Flash N-Shots


In [92]:
# template = PromptTemplate(
#         name="Manually Crafted",
#         definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
#         instruction="Assign the label yes if the comment contains a strong indication of Self-Admitted Technical Debt (SATD); otherwise, assign the label no.",
#         n_shot_template="<EXAMPLE>\nComment: {comment}\nLabel: {label}\n</EXAMPLE>",
#         line_m_before=3,
#         line_n_after=10
#     )

template = PromptTemplate(
        name="Manually Crafted",
        definition="Self-Admitted Technical Debt (SATD) is comment in source code that indicates the need for future improvement or fixes. Developers generally uses  phases like TODO, fixme, etc., as an indication.",
        instruction="Your job is to detect a given comment as Yes for SATD and No for not-SATD. Please don't predict anything else. Do not comment as Yes if developers don't mention explicitly the need for future improvement. Also, label as No if your confidence is low",
        n_shot_template="<EXAMPLE>\nComment: {comment}\nLabel: {label}\n</EXAMPLE>",
        line_m_before=3,
        line_n_after=10
    )

detect_satd('detect', MODEL_CONFIG_GEMINI_2_FLASH, 0, template, FewShotSelectionStrategy.MANUAL_CRAFTED,
            DatasetDict({'train': n_shot_detect_dataset, 'test': Dataset.from_pandas(detect_test_df[:1])}), verbose=False)


Running gemini-2.0-flash
              precision    recall  f1-score   support

          no      1.000     1.000     1.000         1

    accuracy                          1.000         1
   macro avg      1.000     1.000     1.000         1
weighted avg      1.000     1.000     1.000         1



# Detect with `all-MiniLM-L6-v2` Embedding and Logistic Regression

In [44]:
#Logistic Regression
# under_sampler = RandomUnderSampler(sampling_strategy='auto', random_state=42)
# lr_df_all = train_dataset_all.to_pandas()
# indices, _ = under_sampler.fit_resample(lr_df_all.index.values.reshape(-1, 1), lr_df_all['satd'])
# lr_df_resampled = lr_df_all.loc[indices.flatten()]
# lr_df_resampled.reset_index(drop=True)
# lr_dataset = Dataset.from_pandas(lr_df_resampled)

detect_satd('detect', MODEL_CONFIG_SENTENCE_EMBEDDED_LR, 0, template, FewShotSelectionStrategy.MANUAL_CRAFTED,
            DatasetDict({'train': detect_train_balanced_dataset, 'test': detect_test_dataset}))

Running all-MiniLM-L6-v2
              precision    recall  f1-score   support

          no      0.996     0.914     0.953     12655
         yes      0.187     0.860     0.308       292

    accuracy                          0.913     12947
   macro avg      0.592     0.887     0.631     12947
weighted avg      0.978     0.913     0.939     12947

Unknown classification count 0
Unknown classification  []


# Detect with `all-MiniLM-L6-v2` Embedding including Prompt and Logistic Regression

In [49]:
model_config = ModelConfig(name="Sentence Embedded Logistic Regression", architecture="sentence-embedded-regression", uri="all-MiniLM-L6-v2")
template = PromptTemplate(
        name="Manually Crafted",
        definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
        instruction="Assign the label yes if the comment contains a strong indication of Self-Admitted Technical Debt (SATD); otherwise, assign the label no.",
        n_shot_template="Comment: {comment}\nLabel: {label}",
        line_m_before=3,
        line_n_after=10
    )
train_df = detect_train_balanced_df.copy()
test_df = detect_test_df.copy()
for df in [train_df, test_df]:
    df['text'] = df['text'].apply(lambda text: create_prompt(template, [],[], [], [], [], text, '', '', None))

detect_satd('detect', model_config, 0, template, FewShotSelectionStrategy.MANUAL_CRAFTED,
            DatasetDict({'train': Dataset.from_pandas(train_df), 'test': Dataset.from_pandas(test_df)}))

Running all-MiniLM-L6-v2
              precision    recall  f1-score   support

          no      0.995     0.931     0.962     12655
         yes      0.213     0.805     0.337       292

    accuracy                          0.928     12947
   macro avg      0.604     0.868     0.649     12947
weighted avg      0.978     0.928     0.948     12947

Unknown classification count 0
Unknown classification  []


# Detect with SATD Text Mining Based SATD Detector

In [88]:
test_df = detect_test_df[detect_test_df['repository'] != 69]
print(len(test_df))
detect_satd('detect', MODEL_CONFIG_TEXT_MINING_SATD_DETECTOR, 0, template, FewShotSelectionStrategy.MANUAL_CRAFTED,
            DatasetDict({'train': detect_train_balanced_dataset, 'test': Dataset.from_pandas(test_df)}))

12644
Running satd-detector
              precision    recall  f1-score   support

          no      0.994     0.991     0.993     12476
         yes      0.463     0.554     0.504       168

    accuracy                          0.986     12644
   macro avg      0.728     0.772     0.748     12644
weighted avg      0.987     0.986     0.986     12644



# Detect with `text-embedding-004` Embedding and Logistic Regression

In [56]:
model_config = ModelConfig(name="Gemini Embedded Logistic Regression", architecture="gemini-embedded-regression", uri='models/text-embedding-004')
detect_satd('detect', model_config, 0, template, FewShotSelectionStrategy.MANUAL_CRAFTED,
            DatasetDict({'train': detect_train_balanced_dataset, 'test': Dataset.from_pandas(detect_test_df[:2000])}))

Running text-embedding-004
              precision    recall  f1-score   support

          no      0.995     0.934     0.963      1949
         yes      0.243     0.804     0.373        51

    accuracy                          0.931      2000
   macro avg      0.619     0.869     0.668      2000
weighted avg      0.975     0.931     0.948      2000

Unknown classification count 0
Unknown classification  []


# Detect with `text-embedding-004` Embedding including prompt and Logistic Regression

In [70]:
template = PromptTemplate(
        name="Manually Crafted",
        definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
        instruction="Assign the label yes if the comment contains a strong indication of Self-Admitted Technical Debt (SATD); otherwise, assign the label no.",
        n_shot_template="Comment: {comment}\nLabel: {label}",
        line_m_before=3,
        line_n_after=10
    )
train_df = detect_train_balanced_df.copy()
test_df = detect_test_df[:2000].copy()
for df in [train_df, test_df]:
    df['text'] = df['text'].apply(lambda text: create_prompt(template, [],[], [], [], [], text, '', '', None))
model_config = ModelConfig(name="Gemini Embedded Logistic Regression", architecture="gemini-embedded-regression", uri='models/text-embedding-004')
detect_satd('detect', model_config, 0, template, FewShotSelectionStrategy.MANUAL_CRAFTED,
            DatasetDict({'train': Dataset.from_pandas(train_df), 'test': Dataset.from_pandas(test_df)}))

Running text-embedding-004
              precision    recall  f1-score   support

          no      0.993     0.992     0.993      1949
         yes      0.717     0.745     0.731        51

    accuracy                          0.986      2000
   macro avg      0.855     0.869     0.862      2000
weighted avg      0.986     0.986     0.986      2000



# Detect with `models/gemini-embedding-exp-03-07` Embedding and Logistic Regression

In [57]:
model_config = ModelConfig(name="Gemini Embedded Logistic Regression", architecture="gemini-embedded-regression", uri='models/gemini-embedding-exp-03-07')
detect_satd('detect', model_config, 0, template, FewShotSelectionStrategy.MANUAL_CRAFTED,
            DatasetDict({'train': detect_train_balanced_dataset, 'test': Dataset.from_pandas(detect_test_df[:2000])}))

Running gemini-embedding-exp-03-07
              precision    recall  f1-score   support

          no      0.996     0.988     0.992      1949
         yes      0.657     0.863     0.746        51

    accuracy                          0.985      2000
   macro avg      0.827     0.925     0.869      2000
weighted avg      0.988     0.985     0.986      2000

Unknown classification count 0
Unknown classification  []


# Detect with `models/gemini-embedding-exp-03-07` Embedding including Prompt and Logistic Regression


In [59]:
model_config = ModelConfig(name="Gemini Embedded Logistic Regression", architecture="gemini-embedded-regression", uri='models/gemini-embedding-exp-03-07')
template = PromptTemplate(
        name="Manually Crafted",
        definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
        instruction="Assign the label yes if the comment contains a strong indication of Self-Admitted Technical Debt (SATD); otherwise, assign the label no.",
        n_shot_template="Comment: {comment}\nLabel: {label}",
        line_m_before=3,
        line_n_after=10
    )
train_df = detect_train_balanced_df.copy()
test_df = detect_test_df[:200].copy()
for df in [train_df, test_df]:
    df['text'] = df['text'].apply(lambda text: create_prompt(template, [],[], [], [], [], text, '', '', None))
detect_satd('detect', model_config, 0, template, FewShotSelectionStrategy.MANUAL_CRAFTED,
            DatasetDict({'train': Dataset.from_pandas(train_df), 'test': Dataset.from_pandas(test_df)}))


Running gemini-embedding-exp-03-07


RetryError: RetryError[<Future at 0x7303b6f23190 state=finished raised ResourceExhausted>]

# Classify with `all-MiniLM-L6-v2` Embedding and Logistic Regression

In [ ]:
detect_satd('classify', MODEL_CONFIG_SENTENCE_EMBEDDED_LR, 0, template, FewShotSelectionStrategy.MANUAL_CRAFTED,
            DatasetDict({'train': classify_train_dataset, 'test': classify_test_dataset}))

# Classify with `models/gemini-embedding-exp-03-07` Embedding and Logistic Regression

In [252]:
model_config = ModelConfig(name="Gemini Embedded Logistic Regression", architecture="gemini-embedded-regression", uri='models/gemini-embedding-exp-03-07')
detect_satd('classify', model_config, 0, template, FewShotSelectionStrategy.MANUAL_CRAFTED,
            DatasetDict({'train': classify_train_dataset, 'test': classify_test_dataset}))

Running Gemini Embedded Logistic Regression - Manually Crafted 0 -  FewShotSelectionStrategy.MANUAL_CRAFTED


KeyboardInterrupt: 

# Classify with `models/text-embedding-004` Embedding and Logistic Regression

In [ ]:
model_config = ModelConfig(name="Gemini Embedded Logistic Regression", architecture="gemini-embedded-regression", uri='models/text-embedding-004')
detect_satd('classify', model_config, 0, template, FewShotSelectionStrategy.MANUAL_CRAFTED,
            DatasetDict({'train': classify_train_dataset, 'test': classify_test_dataset}))

# Classify with Gemini Flash 2.0 N-Shots


In [64]:
template = PromptTemplate(
        name="Manually Crafted Classification",
        definition="You are an AI model trained to classify Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises.",
        instruction="""Classify Self-Admitted Technical Debt (SATD) in test code into one of the following 16 categories:
        build : Refers to build issues that make tasks harder and more time-consuming. A project’s build process can include unnecessary code or poorly defined dependencies, causing it to run slowly. This is known as build debt.
        code: Refers to poor coding practices, such as bad naming conventions, duplicated code and the use of inefficient algorithms which can negatively impact code readability, maintainability, and performance.
        design: Refers to practices that violate the principles of good object-oriented design, such as high coupling and low cohesion, which can lead to reduced code modularity, flexibility, and maintainability.
        defect: Unresolved known defects that have been identified and require correction.
        skip test: Skipping or disabling tests for any reason.
        temporary fix: A temporary workaround that is intended to address an issue but needs to be replaced with a permanent solution to ensure long-term stability and effectiveness.
        documentation: Documentation Debt refers to the lack, incompleteness, or outdated state of documentation in a project.
        impractical case: Refers to the identification of unexpected or problematic situations that are not anticipated to occur under normal circumstances.
        dependency: Occurs when there is an issue with a dependency.
        superficial: Occurs when a test only covers partial testing.
        how to: Occurs when the solution to a problem is unknown or there is uncertainty about how to proceed, often requiring further investigation or exploration to find a resolution.
        refactor: Refers to debt that mentions about refactoring current code or copying code.
        requirement: Refers to incomplete or insufficient implementation of test or functionality.
        multi: Refers to more debt that contain more than one type of debt.
        other: Any other SATD comments that does not fit into the above classes.
        """,
        n_shot_template="<EXAMPLE>\nComment: {comment}\nLabel: {label}\n</EXAMPLE>",
        line_m_before=3,
        line_n_after=10
    )

detect_satd('detect', MODEL_CONFIG_GEMINI_2_FLASH, 4, template, FewShotSelectionStrategy.MANUAL_CRAFTED,
            DatasetDict({'train': n_shot_detect_dataset, 'test': Dataset.from_pandas(detect_test_df[:2000])}))

Running gemini-2.0-flash
refactor no Failing for // normally we have to parse more data,
// but for simplicity we expect no more chunks...
other no Failing for /* late data output tag */
refactor no Failing for // lets handle the error and store to the results a dummy error DTO
</example> no Failing for // ToUnicode does not set length-overflow errors.
refactor no Failing for // 64-bit trace ID
</example> no Failing for // do the sync
refactor no Failing for // Delete the c family to verify deletes make it over.
other no Failing for // Test uninteresting (empty) panes don't increment the index or otherwise
// modify PaneInfo.
</example> no Failing for //
other no Failing for /*
 * Licensed to the Apache Software Foundation (ASF) under one or more
 * contributor license agreements.  See the NOTICE file distributed with
 * this work for additional information regarding copyright ownership.
 * The ASF licenses this file to You under the Apache License, Version 2.0
 * (the "License"); you 

In [ ]:
if LAST_RUNNING_FILE:
    print('Merging Mismatch')
    ldf = pd.read_csv(LAST_RUNNING_FILE)
    mdf = pd.read_csv('./cache/detect_output.csv')
    ids = mdf['id'].values
    for index, row in ldf.iterrows():
        if row['id'] in ids:
            mdf.loc[mdf['id'] == row['id'], 'label_pred'] = row['label_pred']
        else:
            mdf.loc[len(mdf)] = row
    mdf.sort_values(by=['id'], ascending=True, inplace=True)
    mdf.to_csv('./cache/detect_output.csv', index = False)
    mdf[mdf['label'] != mdf['label_pred']].to_csv('./cache/detect_mismatch.csv', index = False)

In [ ]:
# from comment import CommentRepository
# from db_config import SessionLocal
# from dotenv import load_dotenv
# import os
#
# load_dotenv()
# session = SessionLocal()
# repo = CommentRepository()
# comments = repo.get_comments_with_no_prediction(limit=500)
# ids = []
# texts = []
# labels = []
# for comment in comments:
#     ids.append(comment.id)
#     texts.append(comment.text)
#     labels.append(comment.is_td)
#
# test_dataset = Dataset.from_dict({'text': texts, 'label': labels})
# detection_dataset = DatasetDict({
#     'train': dataset['train'],
#     'test': test_dataset
# })
# # print(detection_dataset)
#
# output = detect_satd(MODEL_CONFIGS[0:1], [3], PROMPT_TEMPLATES[0:1], [FewShotSelectionStrategy.MANUAL_CRAFTED], detection_dataset)
# rc, test_x, test_y, pred_y, unknown_labels = output[0]
# for _,[id, pred] in enumerate(zip(ids,  pred_y)):
#     target_comment = repo.get_comment(id)
#     target_comment.pred_td = True if pred.lower() == 'yes' else False
#     session.merge(target_comment)
#     session.commit()


In [29]:

# target_model = 'gemini'
# comment_df = pd.read_csv('../data/comments.csv')
# test_df = comment_df[comment_df[target_model] is None].sample(frac=1, random_state=42).head(10)
#
# ids = []
# texts = []
# labels = []
# for index, row in df.iterrows():
#     ids.append(row['id'])
#     texts.append(row['text'])
#     labels.append(row[target_model])
#
# test_dataset = Dataset.from_dict({'text': texts, 'label': labels})
# detection_dataset = DatasetDict({
#     'train': dataset['train'],
#     'test': test_dataset
# })
#
# output = detect_satd(MODEL_CONFIGS[0:1], [20], PROMPT_TEMPLATES[0:1], [FewShotSelectionStrategy.MANUAL_CRAFTED],
#                      detection_dataset)
# rc, test_x, test_y, pred_y, unknown_labels = output[0]
# for _, [id, pred] in enumerate(zip(ids, pred_y)):
#     comment_df.loc[df['id'] == id, target_model] = pred
# comment_df.to_csv('../data/comments.csv')
#
